# FFC to CLAHE With FABEMD and MOO

This notebook runs one image through the active pipeline stages from flat-field correction (FFC) through CLAHE. It uses FABEMD decomposition and the repository's current multi-metric optimization objective: `total_score = CII + entropy + EME`.

Parameter naming follows the thesis/pipeline mapping: `gH -> rh`, `beta -> denoise_beta`, `gamma -> gamma correction`, and `clip_limit -> CLAHE`.

In [ ]:
from pathlib import Path
import gc
import itertools
import sys
import time

try:
    import cv2
    import matplotlib.pyplot as plt
    import numpy as np
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'Missing notebook dependency. Run this notebook with the project .venv interpreter.'
    ) from exc

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'image_pipeline.py').exists():
    REPO_ROOT = Path('C:/Users/wonga/repo/pace_implementation')

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from image_pipeline import (
    HAS_CUPY,
    HomomorphicFilter,
    ImageProcessingPipeline,
    PipelineConfig,
    _to_numpy,
)

print(f'Python: {sys.executable}')
print(f'Repo: {REPO_ROOT}')
print(f'OpenCV: {cv2.__version__}')
print(f'CuPy enabled: {HAS_CUPY}')

## Paths and Parameters

Edit these paths if your input image or calibration files live somewhere else.

In [ ]:
DEFAULT_PROJECTION = next(REPO_ROOT.glob('*.tiff'), None)

PROJ_PATH = DEFAULT_PROJECTION
GAIN_PATH = REPO_ROOT / 'datacitra' / 'Gain' / 'Trx' / '90_40_0,50.mdn'
DARK_PATH = REPO_ROOT / 'datacitra' / 'Dark' / 'Trx' / 'dark.mdn'
CALIBRATION_PATH = REPO_ROOT / 'datacitra' / 'Kalibrasi' / 'trx_44_35.npz'
OUTPUT_DIR = REPO_ROOT / 'output' / 'ffc_to_clahe_fabemd_moo'
OUTPUT_PATH = OUTPUT_DIR / 'ffc_to_clahe_fabemd_moo_result.tiff'

required_paths = {
    'projection': PROJ_PATH,
    'gain': GAIN_PATH,
    'dark': DARK_PATH,
    'calibration': CALIBRATION_PATH,
}

for label, path in required_paths.items():
    if path is None or not Path(path).exists():
        raise FileNotFoundError(f'Missing {label} path: {path}')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Projection: {PROJ_PATH}')
print(f'Gain:       {GAIN_PATH}')
print(f'Dark:       {DARK_PATH}')
print(f'Calibration:{CALIBRATION_PATH}')
print(f'Output:     {OUTPUT_PATH}')

In [ ]:
# Compact default grid. Add values here to expand the MOO/grid scoring search.
FIXED_D0 = 40
FIXED_RL = 0.99
GH_VALUES = [2.5]
BETA_VALUES = [1.0]
GAMMA_VALUES = [0.8]
CLIP_LIMIT_VALUES = [3.0]
TILE_GRID_SIZE_VALUES = [(8, 8)]

# FABEMD settings copied from the current batch notebook style.
FABEMD_MAX_SIFT = 1
FABEMD_SD_THRESHOLD = 0.2
FABEMD_MIN_EXTREMA = 5
FABEMD_MAX_BIMFS = 20
FABEMD_WINDOW_SIZE_CAP = 2000
FABEMD_EXTREMA_WINDOW = 3
FABEMD_INITIAL_WINDOW_SIZE = None
FABEMD_WINDOW_GROWTH_RATE = 2.0

PARAMETER_COMBINATIONS = list(itertools.product(
    GH_VALUES,
    BETA_VALUES,
    GAMMA_VALUES,
    CLIP_LIMIT_VALUES,
    TILE_GRID_SIZE_VALUES,
))

config = PipelineConfig(
    proj_img_path=str(PROJ_PATH),
    gain_img_path=str(GAIN_PATH),
    dark_img_path=str(DARK_PATH),
    calibration_path=str(CALIBRATION_PATH),
    output_dir=str(OUTPUT_DIR),
    processing_mode='full',
    homomorphic_method='gaussian',
    decomposition_method='fabemd',
    fabemd_max_sift_iterations=FABEMD_MAX_SIFT,
    fabemd_sd_threshold=FABEMD_SD_THRESHOLD,
    fabemd_min_extrema=FABEMD_MIN_EXTREMA,
    fabemd_max_bimfs=FABEMD_MAX_BIMFS,
    fabemd_window_size_cap=FABEMD_WINDOW_SIZE_CAP,
    fabemd_extrema_window=FABEMD_EXTREMA_WINDOW,
    fabemd_initial_window_size=FABEMD_INITIAL_WINDOW_SIZE,
    fabemd_window_growth_rate=FABEMD_WINDOW_GROWTH_RATE,
    d0_values=[FIXED_D0],
    rh_values=GH_VALUES,
    rl_values=[FIXED_RL],
    gamma_values=GAMMA_VALUES,
    clip_limit_values=CLIP_LIMIT_VALUES,
    tile_grid_size_values=TILE_GRID_SIZE_VALUES,
    denoise_beta=BETA_VALUES[0],
    output_width=4096,
)

pipeline = ImageProcessingPipeline(config)

print(f'MOO/grid combinations: {len(PARAMETER_COMBINATIONS)}')
print(
    'FABEMD config: '
    f'max_sift={FABEMD_MAX_SIFT}, sd={FABEMD_SD_THRESHOLD}, '
    f'min_extrema={FABEMD_MIN_EXTREMA}, max_bimfs={FABEMD_MAX_BIMFS}, '
    f'window_cap={FABEMD_WINDOW_SIZE_CAP}, extrema_window={FABEMD_EXTREMA_WINDOW}, '
    f'initial_window={FABEMD_INITIAL_WINDOW_SIZE}, growth={FABEMD_WINDOW_GROWTH_RATE}'
)

In [ ]:
def elapsed_text(seconds):
    minutes, secs = divmod(seconds, 60)
    hours, minutes = divmod(minutes, 60)
    if hours >= 1:
        return f'{int(hours)}h {int(minutes)}m {secs:.1f}s'
    if minutes >= 1:
        return f'{int(minutes)}m {secs:.1f}s'
    return f'{secs:.1f}s'


def image_summary(name, image):
    arr = _to_numpy(image)
    print(
        f'{name}: shape={arr.shape}, dtype={arr.dtype}, '
        f'min={float(np.min(arr)):.6g}, max={float(np.max(arr)):.6g}'
    )


def show_images(images, figsize=(16, 5)):
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]
    for ax, (title, image) in zip(axes, images):
        arr = _to_numpy(image)
        ax.imshow(arr, cmap='gray')
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


def pipeline_params(params):
    gH, beta, gamma, clip_limit, tile_grid_size = params
    return (FIXED_D0, gH, FIXED_RL, gamma, clip_limit, tile_grid_size), float(beta)


def set_residue_beta(pipeline, beta):
    pipeline.config.denoise_beta = float(beta)
    pipeline.nonlinear_filter.beta = float(beta)


def score_params(pipeline, params, reference_image, bimfs, energies, residue):
    pipeline_param_tuple, beta = pipeline_params(params)
    set_residue_beta(pipeline, beta)
    return pipeline._process_single_params(
        pipeline_param_tuple,
        reference_image,
        bimfs,
        energies,
        residue,
    )


def row_from_result(index, params, result, elapsed_seconds):
    gH, beta, gamma, clip_limit, tile_grid_size = params
    return {
        'rank': None,
        'param_index': index,
        'd0': FIXED_D0,
        'gH_rh': gH,
        'rL': FIXED_RL,
        'beta': beta,
        'gamma': gamma,
        'clip_limit': clip_limit,
        'tile_grid_size': tile_grid_size,
        'cii': result.cii,
        'entropy': result.entropy,
        'eme': result.eme,
        'total_score': result.total_score,
        'elapsed_seconds': elapsed_seconds,
    }

## 1. Load Images and Apply FFC

In [ ]:
start = time.perf_counter()

proj_img, gain_img, dark_img = pipeline.load_images(
    str(PROJ_PATH),
    str(GAIN_PATH),
    str(DARK_PATH),
)
ffc_img = pipeline.apply_ffc(proj_img, gain_img, dark_img)

image_summary('Projection', proj_img)
image_summary('Gain', gain_img)
image_summary('Dark', dark_img)
image_summary('FFC', ffc_img)
print(f'FFC elapsed: {elapsed_text(time.perf_counter() - start)}')

show_images([
    ('Projection', proj_img),
    ('FFC', ffc_img),
])

## 2. Spatial Calibration

In [ ]:
start = time.perf_counter()

calibrated_img = pipeline.apply_spatial_calibration(ffc_img, str(CALIBRATION_PATH))

image_summary('Calibrated', calibrated_img)
print(f'Calibration elapsed: {elapsed_text(time.perf_counter() - start)}')

show_images([
    ('FFC', ffc_img),
    ('Calibrated', calibrated_img),
])

## 3. FABEMD Decomposition

In [ ]:
start = time.perf_counter()

bimfs, energies, residue = pipeline.decompose_image(calibrated_img, method='fabemd')

print(f'BIMFs extracted: {len(bimfs)}')
print(f'Energies: {[round(e, 4) for e in energies]}')
image_summary('FABEMD residue', residue)
print(f'FABEMD elapsed: {elapsed_text(time.perf_counter() - start)}')

preview = [('Calibrated', calibrated_img), ('Residue', residue)]
if bimfs:
    preview.insert(1, ('BIMF 1', bimfs[0]))
show_images(preview, figsize=(18, 5))

## 4. MOO/Grid Scoring

The current project objective is a scalar multi-metric score: `total_score = CII + entropy + EME`. The best row is the highest total score.

In [ ]:
score_rows = []
best_result = None
best_params = None

for index, params in enumerate(PARAMETER_COMBINATIONS, start=1):
    start = time.perf_counter()
    result = score_params(pipeline, params, calibrated_img, bimfs, energies, residue)
    elapsed = time.perf_counter() - start
    row = row_from_result(index, params, result, elapsed)
    score_rows.append(row)

    if best_result is None or result.total_score > best_result.total_score:
        best_result = result
        best_params = params

score_rows = sorted(score_rows, key=lambda row: row['total_score'], reverse=True)
for rank, row in enumerate(score_rows, start=1):
    row['rank'] = rank

print('MOO/grid scores:')
for row in score_rows:
    print(
        f"#{row['rank']} params={row['param_index']} "
        f"gH={row['gH_rh']} beta={row['beta']} gamma={row['gamma']} "
        f"clip={row['clip_limit']} tile={row['tile_grid_size']} | "
        f"CII={row['cii']:.6f}, entropy={row['entropy']:.6f}, "
        f"EME={row['eme']:.6f}, total={row['total_score']:.6f}"
    )

assert best_result is not None
assert best_params is not None
assert best_result.total_score == score_rows[0]['total_score']

print('\nBest parameters:', best_params)
print(f'Best total score: {best_result.total_score:.6f}')

## 5. Expanded Best-Parameter Pipeline: Homomorphic to CLAHE

This cell expands the winning parameter tuple so the code path from FABEMD residue filtering through CLAHE is visible.

In [ ]:
gH, beta, gamma, clip_limit, tile_grid_size = best_params
d0, rh, rl = FIXED_D0, gH, FIXED_RL
set_residue_beta(pipeline, beta)

background_mask = pipeline._reference_background_mask(calibrated_img)

# Homomorphic filtering is applied to the FABEMD residue.
homomorphic_filter = HomomorphicFilter(d0=d0, rh=rh, rl=rl)
filtered_residue = homomorphic_filter.apply(residue, normalize=False)

# Reconstruct from denoised low-energy BIMFs plus beta-weighted filtered residue.
reconstructed = pipeline.nonlinear_filter.denoise(
    bimfs,
    energies,
    filtered_residue,
)

# Gamma correction then CLAHE.
gamma_corrected = pipeline.enhancer.gamma_correction(reconstructed, gamma)
gamma_corrected = pipeline._zero_background(gamma_corrected, background_mask)

clahe_img = pipeline.enhancer.apply_clahe(
    gamma_corrected,
    clip_limit=clip_limit,
    tile_grid_size=tile_grid_size,
)
clahe_img = pipeline._zero_background(clahe_img, background_mask)

mask = np.ones_like(calibrated_img)
cii = pipeline.metrics.calculate_cii(clahe_img, calibrated_img, mask)
entropy = pipeline.metrics.calculate_entropy(clahe_img)
eme = pipeline.metrics.calculate_eme(clahe_img, 4, 4)
total_score = cii + entropy + eme

print(f'Best expanded params: d0={d0}, gH/rh={rh}, rL={rl}, beta={beta}, gamma={gamma}, clip_limit={clip_limit}, tile={tile_grid_size}')
print(f'CII:         {cii:.6f}')
print(f'Entropy:     {entropy:.6f}')
print(f'EME:         {eme:.6f}')
print(f'Total score: {total_score:.6f}')
print(f'Matches MOO best: {np.isclose(total_score, best_result.total_score)}')

show_images([
    ('Filtered residue', filtered_residue),
    ('Reconstructed', reconstructed),
    ('Gamma corrected', gamma_corrected),
    ('CLAHE final', clahe_img),
], figsize=(20, 5))

## 6. Normalize, Resize, and Save

In [ ]:
final_image = pipeline.normalize_and_resize(clahe_img, target_width=config.output_width)
pipeline.save_image(final_image, str(OUTPUT_PATH))

image_summary('Final saved image', final_image)
print(f'Saved to: {OUTPUT_PATH}')

show_images([
    ('Projection', proj_img),
    ('Calibrated', calibrated_img),
    ('Final CLAHE', final_image),
], figsize=(18, 5))

del mask
gc.collect()